In [ ]:
# Cell 1: 필수 라이브러리 설치
# Playwright가 설치되어 있지 않다면 아래 셀을 먼저 실행하세요.
%pip install playwright
!playwright install chromium


In [1]:
# Cell 2: 라이브러리 import 및 설정
import sys
import tempfile
import subprocess
import os
import json
import time

# 노트북에서 시각 디버깅을 원하면 False로 설정하세요
NOTEBOOK_HEADLESS = True

# 수집할 트렌드 개수
MAX_TRENDS = 30

# 쿠키 파일 경로
COOKIE_FILE = "twitter_cookies.json"

print(f"📦 최대 수집 개수: {MAX_TRENDS}개")
print(f"🍪 쿠키 파일: {COOKIE_FILE}\n")


📦 최대 수집 개수: 30개
🍪 쿠키 파일: twitter_cookies.json



In [2]:
# Cell 3: 쿠키 저장 스크립트 생성 함수

def create_save_cookie_script(cookie_file):
    """
    쿠키 저장 스크립트를 생성합니다.
    
    Args:
        cookie_file: 쿠키 파일 경로
        
    Returns:
        str: 쿠키 저장 스크립트 문자열
    """
    script = f"""from playwright.sync_api import sync_playwright
import json
import os
import time

cookie_file = {cookie_file!r}

print("🌐 브라우저를 엽니다...")
print("📝 다음 단계를 따라주세요:")
print("   1. 브라우저에서 X.com 로그인 페이지가 열립니다")
print("   2. 수동으로 로그인을 완료하세요")
print("   3. 로그인이 완료되면 이 터미널에서 Enter를 누르세요")
print()

with sync_playwright() as p:
    # headless=False로 브라우저를 표시
    browser = p.chromium.launch(
        headless=False,
        args=[
            "--disable-blink-features=AutomationControlled",
            "--disable-dev-shm-usage",
            "--no-sandbox",
            "--disable-setuid-sandbox"
        ]
    )
    
    # 봇 감지 우회를 위한 컨텍스트 설정
    viewport_size = {{"width": 1920, "height": 1080}}
    # CORS 문제를 피하기 위해 최소한의 헤더만 사용
    headers_dict = {{"Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7", "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8"}}
    context = browser.new_context(
        user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        viewport=viewport_size,
        java_script_enabled=True,
        bypass_csp=True,
        ignore_https_errors=False,
        extra_http_headers=headers_dict
    )
    
    page = context.new_page()
    
    # 봇 감지 우회 스크립트 주입
    anti_bot_script = "Object.defineProperty(navigator, 'webdriver', {{ get: () => undefined }}); window.chrome = {{ runtime: {{}} }}; Object.defineProperty(navigator, 'plugins', {{ get: () => [1, 2, 3, 4, 5] }}); Object.defineProperty(navigator, 'languages', {{ get: () => ['ko-KR', 'ko', 'en-US', 'en'] }}); const originalQuery = window.navigator.permissions.query; window.navigator.permissions.query = (parameters) => (parameters.name === 'notifications' ? Promise.resolve({{ state: Notification.permission }}) : originalQuery(parameters)); const getParameter = WebGLRenderingContext.getParameter; WebGLRenderingContext.prototype.getParameter = function(parameter) {{ if (parameter === 37445) {{ return 'Intel Inc.'; }} if (parameter === 37446) {{ return 'Intel Iris OpenGL Engine'; }} return getParameter(parameter); }};"
    page.add_init_script(anti_bot_script)
    
    # 로그인 페이지로 이동
    login_url = "https://x.com/i/flow/login"
    print(f"접속 중: {{login_url}}")
    page.goto(login_url, wait_until="domcontentloaded", timeout=60000)
    time.sleep(5)  # 페이지 완전 로드 대기
    
    print("\\\\n⚠️ Google 로그인 사용 시 팝업 창이 열릴 수 있습니다.")
    print("   팝업 창에서 로그인을 완료하면 자동으로 X.com으로 돌아옵니다.")
    print("   또는 이메일/비밀번호 로그인을 사용하는 것을 권장합니다.\\\\n")
    
    # 팝업 창 처리 (Google OAuth용)
    def handle_popup(popup):
        print(f"팝업 창 감지: {{popup.url}}")
        try:
            popup.wait_for_load_state("networkidle", timeout=30000)
            print("팝업 창 로드 완료")
        except:
            pass
    
    # 팝업 이벤트 리스너 등록
    context.on("page", handle_popup)
    
    # 사용자가 로그인할 때까지 자동 감지
    print("로그인을 진행하세요...")
    print("(Google 로그인 사용 시 팝업 창에서 로그인 후 자동으로 돌아옵니다)")
    print("로그인 완료를 자동으로 감지합니다 (최대 5분 대기)...\\\\n")
    
    # 로그인 완료 자동 감지
    max_wait_time = 300  # 5분
    check_interval = 3  # 3초마다 확인
    start_time = time.time()
    login_completed = False
    elapsed = 0  # 초기화
    
    while time.time() - start_time < max_wait_time:
        elapsed = int(time.time() - start_time)
        time.sleep(check_interval)
        
        try:
            current_url = page.url
            page_content = page.content()
            
            # 로그인 완료 확인 조건
            is_not_login_page = "login" not in current_url.lower() and "flow" not in current_url.lower()
            has_no_login_ui = "Log in" not in page_content and "Sign up" not in page_content
            
            if is_not_login_page and has_no_login_ui:
                # 추가 확인: 홈 피드나 트렌드 관련 요소가 있는지
                if "explore" in current_url.lower() or "home" in current_url.lower() or len(page.query_selector_all("nav")) > 0:
                    print("\\\\n✅ 로그인 완료 감지! (" + str(elapsed) + "초 경과)")
                    login_completed = True
                    break
            
            # 진행 상황 표시 (30초마다)
            if elapsed % 30 == 0 and elapsed > 0:
                print("로그인 대기 중... (" + str(elapsed) + "초 경과, 최대 " + str(max_wait_time) + "초)")
        except Exception as e:
            # 페이지 접근 오류는 무시하고 계속 대기
            pass
    
    # 팝업 창이 열려있는지 확인하고 닫기
    if len(context.pages) > 1:
        print("\\\\n팝업 창 닫는 중...")
        for popup_page in context.pages[1:]:
            try:
                popup_page.close()
            except:
                pass
    
    # 자동 감지 실패 시 수동 입력 요청
    if not login_completed:
        print("\\\\n⚠️ 자동 감지 실패 (" + str(max_wait_time) + "초 경과)")
        print("   로그인이 완료되었다면 브라우저를 확인해주세요.")
        print("   수동으로 Enter를 눌러 계속 진행하거나, 브라우저를 닫고 다시 시도하세요.")
        try:
            input("로그인을 완료한 후 Enter를 누르세요 (또는 Ctrl+C로 취소)...")
        except KeyboardInterrupt:
            print("\\\\n취소되었습니다.")
            browser.close()
            exit(1)
    
    # 메인 페이지가 로그인 완료되었는지 최종 확인
    time.sleep(2)
    current_url = page.url
    print(f"현재 URL: {{current_url}}")
    
    # 로그인 확인: 트렌드 페이지로 이동 시도
    print("\\\\n로그인 상태 확인 중...")
    try:
        page.goto("https://x.com/explore/tabs/trending", wait_until="domcontentloaded", timeout=60000)
        time.sleep(5)  # 페이지 로드 대기
    except Exception as e:
        print(f"페이지 이동 중 오류: {{e}}")
        print("현재 페이지에서 로그인 상태를 확인합니다...")
    
    # 로그인 페이지로 리다이렉트되었는지 확인
    current_url = page.url
    page_content = page.content()
    
    if "login" in current_url.lower() or "flow" in current_url.lower():
        print(f"⚠️ 로그인이 완료되지 않은 것 같습니다. (현재 URL: {{current_url}})")
        print("   다음을 시도해보세요:")
        print("   1. 이메일/비밀번호 로그인 사용 (Google 로그인 대신)")
        print("   2. 브라우저에서 직접 로그인 후 쿠키 저장")
        print("   3. 다시 시도")
        browser.close()
        exit(1)
    
    # 로그인 성공 확인 (홈 피드나 트렌드가 보이는지)
    if "Log in" in page_content or "Sign up" in page_content:
        print("⚠️ 로그인 페이지가 여전히 표시되고 있습니다.")
        print("   로그인을 다시 시도해주세요.")
        browser.close()
        exit(1)
    
    # 쿠키 저장
    cookies = context.cookies()
    with open(cookie_file, "w", encoding="utf-8") as f:
        json.dump(cookies, f, ensure_ascii=False, indent=2)
    
    print(f"\\\\n✅ 쿠키 저장 완료: {{cookie_file}}")
    print(f"📦 저장된 쿠키 개수: {{len(cookies)}}개")
    print("\\\\n이제 Cell 5를 실행하여 크롤링할 수 있습니다!")
    
    browser.close()
"""
    return script


In [3]:
# Cell 4: 쿠키 저장 실행
# 브라우저를 열고 수동으로 로그인한 후 쿠키를 저장합니다.
# 이 셀은 처음 한 번만 실행하면 됩니다. 쿠키가 만료되면 다시 실행하세요.

print("▶ 트위터 쿠키 저장을 시작합니다...\\n")

# 쿠키 저장 스크립트 생성
save_cookie_script = create_save_cookie_script(COOKIE_FILE)

# 임시 파일에 스크립트 기록 후 실행
fd, path = tempfile.mkstemp(suffix="_save_twitter_cookies.py")
os.close(fd)
with open(path, "w", encoding="utf-8") as f:
    f.write(save_cookie_script)

try:
    completed = subprocess.run(
        [sys.executable, path], capture_output=False, text=True, check=False
    )
finally:
    try:
        os.remove(path)
    except Exception:
        pass


▶ 트위터 쿠키 저장을 시작합니다...\n


In [4]:
# Cell 5: 크롤링 스크립트 생성 함수

def create_twitter_crawl_script(headless, max_trends, cookie_file):
    """
    크롤링 스크립트를 생성합니다.
    
    Args:
        headless: 헤드리스 모드 여부
        max_trends: 최대 수집할 트렌드 개수
        cookie_file: 쿠키 파일 경로
        
    Returns:
        str: 크롤링 스크립트 문자열
    """
    script = f"""from playwright.sync_api import sync_playwright
import json
import time
import os

headless = {headless!r}
max_trends = {max_trends}
cookie_file = {cookie_file!r}

trends = []
found_keywords = set()

# 쿠키 파일 확인
if not os.path.exists(cookie_file):
    print(f"❌ 쿠키 파일을 찾을 수 없습니다: {{cookie_file}}")
    print("   먼저 Cell 4를 실행하여 쿠키를 저장하세요.")
    exit(1)

with sync_playwright() as p:
    # 봇 감지 우회를 위한 브라우저 설정
    browser = p.chromium.launch(
        headless=headless,
        args=[
            "--disable-blink-features=AutomationControlled",
            "--disable-dev-shm-usage",
            "--no-sandbox",
            "--disable-setuid-sandbox"
        ]
    )
    
    # 봇 감지 우회를 위한 컨텍스트 설정
    viewport_size = {{"width": 1920, "height": 1080}}
    # CORS 문제를 피하기 위해 최소한의 헤더만 사용
    headers_dict = {{"Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7", "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8"}}
    context = browser.new_context(
        user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        viewport=viewport_size,
        java_script_enabled=True,
        bypass_csp=True,
        ignore_https_errors=False,
        extra_http_headers=headers_dict
    )
    
    # 저장된 쿠키 로드
    print(f"📂 쿠키 파일 로드 중: {{cookie_file}}")
    with open(cookie_file, "r", encoding="utf-8") as f:
        cookies = json.load(f)
    context.add_cookies(cookies)
    print(f"✅ {{len(cookies)}}개의 쿠키를 로드했습니다.")
    
    page = context.new_page()
    
    # 봇 감지 우회 스크립트 주입
    anti_bot_script = "Object.defineProperty(navigator, 'webdriver', {{ get: () => undefined }}); window.chrome = {{ runtime: {{}} }}; Object.defineProperty(navigator, 'plugins', {{ get: () => [1, 2, 3, 4, 5] }}); Object.defineProperty(navigator, 'languages', {{ get: () => ['ko-KR', 'ko', 'en-US', 'en'] }}); const originalQuery = window.navigator.permissions.query; window.navigator.permissions.query = (parameters) => (parameters.name === 'notifications' ? Promise.resolve({{ state: Notification.permission }}) : originalQuery(parameters)); const getParameter = WebGLRenderingContext.getParameter; WebGLRenderingContext.prototype.getParameter = function(parameter) {{ if (parameter === 37445) {{ return 'Intel Inc.'; }} if (parameter === 37446) {{ return 'Intel Iris OpenGL Engine'; }} return getParameter(parameter); }};"
    page.add_init_script(anti_bot_script)
    
    # 트위터 트렌딩 페이지 접속
    url = "https://x.com/explore/tabs/trending"
    print(f"\\\\n접속 중: {{url}}")
    
    try:
        page.goto(url, wait_until="networkidle", timeout=60000)
        time.sleep(5)  # JavaScript 실행 대기
        
        # 쿠키 만료 확인
        page_content = page.content()
        if "Log in" in page_content or "Sign up" in page_content or "login" in page.url.lower():
            print("\\\\n⚠️ 쿠키가 만료되었거나 로그인이 필요합니다.")
            print("   Cell 4를 다시 실행하여 쿠키를 갱신하세요.")
            browser.close()
            exit(1)
        
        # 트렌드 콘텐츠가 로드될 때까지 대기
        print("트렌드 콘텐츠 로드 대기 중...")
        try:
            # 트렌드 항목이 나타날 때까지 대기
            page.wait_for_selector("span.css-1jxf684", timeout=20000)
            print("✅ 트렌드 페이지 로드 완료")
        except:
            print("⚠️ 기본 선택자 대기 실패, 계속 진행...")
        
        # 추가 대기 (동적 콘텐츠 로드)
        time.sleep(3)
        
        # 스크롤하여 더 많은 트렌드 로드
        print("스크롤하여 더 많은 트렌드 로드 중...")
        for scroll_idx in range(10):
            page.evaluate("window.scrollBy(0, window.innerHeight)")
            time.sleep(2)
            if scroll_idx % 3 == 0:
                time.sleep(2)
        
        page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
        time.sleep(3)
        
        # 방법 1: 제공된 CSS 클래스 구조를 이용한 추출
        print("\\\\n1. CSS 클래스 구조를 이용한 트렌드 추출 중...")
        
        # UI 텍스트 제외 목록 (강화)
        excluded_ui_texts = [
            "Home", "Explore", "Notifications", "Messages", "Grok", "Lists", "Bookmarks",
            "Communities", "Premium", "Profile", "More", "Post", "Trending", "For You",
            "Following", "Search", "News", "Sports", "Entertainment", "See new posts",
            "To view keyboard shortcuts", "View keyboard shortcuts", "Promoted by",
            "Only on X", "Trending in", "posts", "게시물", "트윗", "·", "1", "2", "3", "4", "5",
            "홈", "탐색", "알림", "메시지", "검색", "더보기", "게시", "트렌드"
        ]
        
        # data-testid="trend" 요소에서만 추출하도록 변경
        trend_elements = page.query_selector_all('[data-testid="trend"]')
        
        for trend_elem in trend_elements:
            try:
                # span.css-1jxf684.r-bcqeeo.r-1ttztb7.r-qvutc0.r-poiln3 클래스를 가진 모든 요소 찾기
                spans = trend_elem.query_selector_all("span.css-1jxf684.r-bcqeeo.r-1ttztb7.r-qvutc0.r-poiln3")
                
                for span in spans:
                    text = span.inner_text().strip()
                    
                    if text and len(text) > 1 and len(text) < 100:
                        # 중복 제거 및 필터링 (UI 텍스트 제외)
                        if (text not in found_keywords and
                            not any(excluded in text for excluded in excluded_ui_texts) and
                            not text.startswith("http") and
                            not text.startswith("@") and
                            not text.isdigit() and
                            text != "·" and
                            len(text.split()) <= 15):
                            
                            found_keywords.add(text)
                            trends.append({{
                                "keyword": text
                            }})
                            
                            if len(trends) >= max_trends:
                                break
                    
                    if len(trends) >= max_trends:
                        break
            except Exception as e:
                continue
        
        # 방법 2: CSS 클래스 구조를 이용한 추가 추출 (백업)
        if len(trends) < max_trends:
            print(f"\\\\n2. CSS 클래스 구조를 이용한 추가 트렌드 추출 중... (현재 {{len(trends)}}개)")
            
            # data-testid="trend" 내부의 모든 span 추출
            trend_containers = page.query_selector_all('[data-testid="trend"]')
            for container in trend_containers:
                try:
                    # container 내부의 특정 클래스를 가진 모든 span 찾기
                    spans = container.query_selector_all("span.css-1jxf684.r-bcqeeo.r-1ttztb7.r-qvutc0.r-poiln3")
                    
                    for span in spans:
                        text = span.inner_text().strip()
                        
                        # 필터링
                        if (text and len(text) > 1 and len(text) < 100 and
                            text not in found_keywords and
                            not any(excluded in text for excluded in excluded_ui_texts) and
                            not text.startswith("http") and
                            not text.startswith("@") and
                            not text.isdigit() and
                            text != "·" and
                            len(text.split()) <= 15):
                            
                            found_keywords.add(text)
                            trends.append({{
                                "keyword": text
                            }})
                            
                            if len(trends) >= max_trends:
                                break
                        
                        if len(trends) >= max_trends:
                            break
                except:
                    continue
        
        # 최종 결과 정리 및 추가 필터링
        # UI 텍스트 최종 필터링
        final_excluded = [
            "Log in", "Sign up", "Don't miss what's happening", 
            "People on X are the first to know", "New to X?",
            "To view keyboard shortcuts", "View keyboard shortcuts",
            "Home", "Explore", "Notifications", "Messages", "Grok", "Lists",
            "Bookmarks", "Communities", "Premium", "Profile", "More", "Post",
            "Trending", "For You", "News", "Sports", "Entertainment",
            "See new posts", "Promoted by", "Only on X", "·", "1", "2", "3", "4", "5"
        ]
        
        filtered_trends = []
        for trend in trends:
            keyword = trend.get("keyword", "")
            # UI 텍스트가 아닌 것만 포함
            if (keyword and 
                not any(excluded in keyword for excluded in final_excluded) and
                not keyword.isdigit() and
                keyword != "·" and
                len(keyword) > 1):
                filtered_trends.append({{"keyword": keyword}})
        
        trends = filtered_trends[:max_trends]
        
        if len(trends) == 0:
            print("\\\\n⚠️ 실제 트렌드를 찾을 수 없습니다. 쿠키가 만료되었을 수 있습니다.")
            print("   Cell 4를 다시 실행하여 쿠키를 갱신하세요.")
        else:
            print(f"\\\\n✅ 총 {{len(trends)}}개 트렌드 키워드 수집 완료")
        
    except Exception as e:
        print(f"⚠ 크롤링 오류: {{e}}")
        import traceback
        traceback.print_exc()
    
    finally:
        browser.close()
    
    # 결과 저장
    result = {{
        "total_trends": len(trends),
        "source": "x.com/explore/tabs/trending",
        "collected_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "trends": trends
    }}
    
    # JSON 파일로 저장
    output_json = "twitter_trends_results.json"
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    
    print(f"✅ JSON 파일 저장 완료: {{output_json}}")
    
    # 결과 미리보기
    print(f"\\\\n📋 수집된 트렌드 키워드 미리보기 (처음 15개):")
    for idx, trend in enumerate(trends[:15], 1):
        print(f"  {{idx}}. {{trend['keyword']}}")
"""
    return script


In [5]:
# Cell 6: 크롤링 실행 및 결과 저장
# 저장된 쿠키를 사용하여 트렌드 30개를 수집하여 JSON 파일로 저장합니다.
# 먼저 Cell 4를 실행하여 쿠키를 저장해야 합니다.

print("▶ 트위터(X.com) 트렌드 크롤링을 시작합니다...")
print(f"📦 최대 수집 개수: {MAX_TRENDS}개\\n")

# 크롤링 스크립트 생성
crawl_script = create_twitter_crawl_script(NOTEBOOK_HEADLESS, MAX_TRENDS, COOKIE_FILE)

# 임시 파일에 스크립트 기록 후 실행
fd, path = tempfile.mkstemp(suffix="_crawl_twitter_trends.py")
os.close(fd)
with open(path, "w", encoding="utf-8") as f:
    f.write(crawl_script)

try:
    completed = subprocess.run(
        [sys.executable, path], capture_output=True, text=True, check=False
    )
    print(completed.stdout)
    if completed.returncode != 0:
        print("--- 프로세스가 에러로 종료되었습니다 (stderr) ---")
        print(completed.stderr)
finally:
    try:
        os.remove(path)
    except Exception:
        pass


▶ 트위터(X.com) 트렌드 크롤링을 시작합니다...
📦 최대 수집 개수: 30개\n
📂 쿠키 파일 로드 중: twitter_cookies.json
✅ 65개의 쿠키를 로드했습니다.
\n접속 중: https://x.com/explore/tabs/trending
트렌드 콘텐츠 로드 대기 중...
✅ 트렌드 페이지 로드 완료
스크롤하여 더 많은 트렌드 로드 중...
\n1. CSS 클래스 구조를 이용한 트렌드 추출 중...
\n✅ 총 30개 트렌드 키워드 수집 완료
✅ JSON 파일 저장 완료: twitter_trends_results.json
\n📋 수집된 트렌드 키워드 미리보기 (처음 15개):
  1. 라이즈 Fame
  2. 함께라면 우린 뭐든 이뤄내
  3. 오늘 오후 6시 라이즈 Fame 킵고잉!!!!
  4. 이번주도
  5. 중간정도
  6. 어른이세
  7. #레진시그
  8. #큥이버블
  9. 그림그리는 트친
  10. 그릴때 즐기는 노동요
  11. 최애의 반응
  12. 이젠 탈덕
  13. 밀리언셀러
  14. 로만 이루어진 데스게임커뮤
  15. #OurDestinyWOOZIday

